## 🎯 Learning Objectives
* Understand the fundamental concepts of Actor-Critic methods, including their advantages over pure policy-based or value-based approaches.
* Differentiate between A2C (Advantage Actor-Critic) and PPO (Proximal Policy Optimization) in terms of their core mechanisms and stability improvements.
* Implement a simplified PPO agent using PyTorch and Gymnasium to solve a classic control problem.
* Analyze the performance characteristics and hyperparameter sensitivities of Actor-Critic algorithms.
* Identify real-world applications of Actor-Critic methods, particularly in the context of LLM alignment and advanced training.


# Actor-Critic Methods: A2C and PPO

Welcome to Lesson FT03-L06, where we delve into the powerful world of Actor-Critic methods, a cornerstone of modern Deep Reinforcement Learning. These algorithms elegantly combine the strengths of both policy-based (Actor) and value-based (Critic) approaches, offering a robust framework for learning complex behaviors.

## The Core Idea: Actor and Critic

Imagine a theatrical production. You have a **Director** (the Actor) who decides what actions the performers should take on stage. This director's goal is to maximize the audience's applause (rewards). However, the director doesn't know *how good* their decisions are until the show is over. This is akin to a pure policy-based method, which learns directly from episode returns.

Now, introduce a **Critic** – a seasoned theater reviewer. The critic watches the performance and provides immediate feedback on how good a particular scene or action was, even before the entire play concludes. This feedback helps the director adjust their decisions more quickly and efficiently. The critic learns to predict the future success (value) of the play from any given point.

In Reinforcement Learning:

*   **Actor**: This is a neural network that learns the **policy** $\pi(a|s)$, which maps states to actions. It decides *what to do*.
*   **Critic**: This is another neural network that learns the **value function** $V(s)$ or $Q(s,a)$. It estimates *how good* a given state or state-action pair is.

Together, the Actor proposes actions, and the Critic evaluates them, guiding the Actor towards better policies. This synergy allows for more stable and efficient learning compared to their standalone counterparts.

## A2C: Advantage Actor-Critic

A2C, or Advantage Actor-Critic, is a synchronous variant of the A3C (Asynchronous Advantage Actor-Critic) algorithm. Its key innovation lies in using the **Advantage function** to guide the policy updates.

### Why Advantage?

Traditional policy gradient methods use the total return $R_t$ (or $Q(s,a)$) to update the policy. However, $R_t$ can be noisy and doesn't tell us if an action was *better than average* for that state. The Advantage function addresses this:

$$A(s,a) = Q(s,a) - V(s)$$ 

Where:
*   $Q(s,a)$ is the action-value function (expected return from state $s$ taking action $a$).
*   $V(s)$ is the state-value function (expected return from state $s$ following the current policy).

In practice, $Q(s,a)$ is often approximated by $R_t + \gamma V(s_{t+1})$, leading to the TD-error as an estimate for the advantage:

$$A(s,a) \approx R_t + \gamma V(s_{t+1}) - V(s_t)$$ 

This advantage tells us *how much better* an action $a$ was compared to the average expected outcome from state $s$. If $A(s,a) > 0$, the action was better than average, and the Actor should be encouraged to take it more often. If $A(s,a) < 0$, the action was worse, and the Actor should be discouraged.

**A2C Mechanism:**
1.  The Actor proposes an action $a_t$ given state $s_t$.
2.  The Critic estimates $V(s_t)$ and $V(s_{t+1})$.
3.  The Advantage $A_t$ is calculated using the observed reward $R_t$ and the Critic's estimates.
4.  The Actor's policy is updated in the direction of higher advantage.
5.  The Critic's value function is updated to minimize the difference between its estimate $V(s_t)$ and the actual observed return (or TD target).

A2C provides a solid foundation, but policy updates can still be unstable if the learning rate is too high, leading to large, destructive changes in the policy.

## PPO: Proximal Policy Optimization

PPO, or Proximal Policy Optimization, is one of the most popular and robust RL algorithms in 2026, widely used in everything from robotics to aligning large language models (LLMs). It builds upon Actor-Critic principles but introduces a clever mechanism to prevent overly aggressive policy updates.

### The Problem with Large Policy Updates

In policy gradient methods, a small change in parameters can sometimes lead to a drastically different policy, which can destabilize training. Imagine our Director suddenly deciding to change the entire play's script based on one bad review – it's too risky!

### PPO's Solution: Clipped Surrogate Objective

PPO addresses this by introducing a **clipped surrogate objective function**. This objective encourages policy updates but *clips* the advantage if the new policy deviates too much from the old policy. It's like putting a leash on the Director, allowing them to experiment but preventing them from straying too far from the successful script.

The PPO objective function (simplified) looks like this:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t) \right]$$

Where:
*   $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$ is the ratio of the new policy probability to the old policy probability for the taken action.
*   $A_t$ is the advantage estimate.
*   $\epsilon$ is a small hyperparameter (e.g., 0.1 or 0.2) that defines the clipping range.

**How the clipping works:**
*   If the new policy is much better than the old one (large $r_t(\theta)$ and positive $A_t$), the ratio $r_t(\theta)$ is clipped at $1+\epsilon$. This means the policy update is limited, preventing it from taking too large a step.
*   If the new policy is much worse (small $r_t(\theta)$ and negative $A_t$), the ratio $r_t(\theta)$ is clipped at $1-\epsilon$. This also limits the penalty, preventing the policy from being overly punished for a bad action.

By clipping the objective, PPO ensures that policy updates are *proximal* (close) to the previous policy, leading to more stable and reliable learning. This stability, combined with its relative simplicity, has made PPO the go-to algorithm for many complex RL tasks, including the fine-tuning of LLMs through Reinforcement Learning from Human Feedback (RLHF).

In the following code example, we will implement a simplified PPO agent to demonstrate these concepts in action.


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import collections

# Ensure reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# 1. Define Actor and Critic Networks
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, state):
        logits = self.net(state)
        return logits

class Critic(nn.Module):
    def __init__(self, state_dim):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state):
        value = self.net(state)
        return value

# 2. Define the PPO Agent
class PPOAgent:
    def __init__(self, state_dim, action_dim, lr_actor=3e-4, lr_critic=1e-3, gamma=0.99, clip_epsilon=0.2, ppo_epochs=10, mini_batch_size=64):
        self.gamma = gamma
        self.clip_epsilon = clip_epsilon
        self.ppo_epochs = ppo_epochs
        self.mini_batch_size = mini_batch_size

        self.actor = Actor(state_dim, action_dim)
        self.critic = Critic(state_dim)

        self.optimizer_actor = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.optimizer_critic = optim.Adam(self.critic.parameters(), lr=lr_critic)

        self.memory = collections.deque() # Stores (state, action, reward, next_state, done, log_prob)

    def select_action(self, state):
        state = torch.FloatTensor(state).unsqueeze(0) # Add batch dimension
        logits = self.actor(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob.item()

    def store_transition(self, state, action, reward, next_state, done, log_prob):
        self.memory.append((state, action, reward, next_state, done, log_prob))

    def clear_memory(self):
        self.memory.clear()

    def compute_advantages(self, states, rewards, next_states, dones):
        # Convert to tensors
        states_t = torch.FloatTensor(states)
        rewards_t = torch.FloatTensor(rewards).unsqueeze(1)
        next_states_t = torch.FloatTensor(next_states)
        dones_t = torch.FloatTensor(dones).unsqueeze(1)

        # Calculate values
        values = self.critic(states_t)
        next_values = self.critic(next_states_t)

        # Calculate TD targets and advantages (simplified GAE for brevity)
        td_targets = rewards_t + self.gamma * next_values * (1 - dones_t)
        advantages = td_targets - values

        return advantages.detach(), td_targets.detach()

    def update(self):
        if not self.memory:
            return

        # Extract data from memory
        states, actions, rewards, next_states, dones, old_log_probs = zip(*self.memory)
        states = np.array(states)
        actions = torch.LongTensor(actions).unsqueeze(1)
        old_log_probs = torch.FloatTensor(old_log_probs).unsqueeze(1)

        # Compute advantages and TD targets
        advantages, td_targets = self.compute_advantages(states, rewards, next_states, dones)

        # Convert states to tensor for PPO epochs
        states_t = torch.FloatTensor(states)

        # PPO optimization loop
        for _ in range(self.ppo_epochs):
            # Create mini-batches
            indices = np.arange(len(states))
            np.random.shuffle(indices)
            for start in range(0, len(states), self.mini_batch_size):
                end = start + self.mini_batch_size
                batch_indices = indices[start:end]

                batch_states = states_t[batch_indices]
                batch_actions = actions[batch_indices]
                batch_old_log_probs = old_log_probs[batch_indices]
                batch_advantages = advantages[batch_indices]
                batch_td_targets = td_targets[batch_indices]

                # Actor update
                logits = self.actor(batch_states)
                dist = Categorical(logits=logits)
                new_log_probs = dist.log_prob(batch_actions.squeeze(1)).unsqueeze(1)

                ratio = torch.exp(new_log_probs - batch_old_log_probs)
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1.0 - self.clip_epsilon, 1.0 + self.clip_epsilon) * batch_advantages
                actor_loss = -torch.min(surr1, surr2).mean()

                # Critic update
                current_values = self.critic(batch_states)
                critic_loss = nn.MSELoss()(current_values, batch_td_targets)

                # Backpropagation
                self.optimizer_actor.zero_grad()
                actor_loss.backward(retain_graph=True) # retain_graph for critic loss if needed, or separate backward calls
                self.optimizer_actor.step()

                self.optimizer_critic.zero_grad()
                critic_loss.backward()
                self.optimizer_critic.step()

        self.clear_memory()

# 3. Training Loop
if __name__ == '__main__':
    env_name = 'CartPole-v1'
    env = gym.make(env_name)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    agent = PPOAgent(state_dim, action_dim)

    num_episodes = 500
    max_timesteps_per_episode = 200 # CartPole-v1 is solved at 200
    update_timestep = 2048 # Update policy every N timesteps (batch size for PPO)
    timestep_count = 0
    episode_rewards = []

    print(f"Training PPO Agent on {env_name} for {num_episodes} episodes...")

    for episode in range(1, num_episodes + 1):
        state, _ = env.reset(seed=SEED + episode)
        episode_reward = 0

        for t in range(max_timesteps_per_episode):
            action, log_prob = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.store_transition(state, action, reward, next_state, done, log_prob)

            state = next_state
            episode_reward += reward
            timestep_count += 1

            # Perform PPO update if enough timesteps collected
            if timestep_count % update_timestep == 0:
                agent.update()

            if done:
                break

        episode_rewards.append(episode_reward)

        # Log progress
        if episode % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f"Episode {episode}/{num_episodes}, Last 10 Avg Reward: {avg_reward:.2f}")

            # CartPole-v1 is considered solved if average reward over 100 episodes is 195
            if len(episode_rewards) >= 100 and np.mean(episode_rewards[-100:]) >= 195:
                print(f"Environment solved in {episode} episodes!")
                break

    env.close()
    print("Training complete.")

    # Optional: Visualize learning curve
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.plot(episode_rewards)
    plt.title('PPO Training Rewards on CartPole-v1')
    plt.xlabel('Episode')
    plt.ylabel('Total Reward')
    plt.grid(True)
    plt.show()


### Interpreting the Code Output and Performance

The code above implements a simplified PPO agent to solve the `CartPole-v1` environment. When you run it, you should observe the following:

1.  **Increasing Rewards**: The primary indicator of successful learning is a steady increase in the "Last 10 Avg Reward" over episodes. For `CartPole-v1`, the environment is considered solved when the average reward over 100 consecutive episodes reaches 195. You should see the agent achieve this within a few hundred episodes.
2.  **Convergence**: The agent's learning curve (if plotted) should show a clear upward trend, eventually plateauing around the maximum possible reward (200 for CartPole-v1).

### Performance Trade-offs and Considerations

*   **A2C vs. PPO Stability**: While A2C is simpler to implement, PPO's clipped surrogate objective provides significantly more stable training. This stability is crucial for complex environments and when dealing with large neural networks, as it prevents catastrophic policy shifts.
*   **On-Policy Nature**: Both A2C and PPO are on-policy algorithms, meaning they learn from data collected by the *current* policy. This makes them less data-efficient than off-policy methods (like DQN or SAC) because old data cannot be reused extensively. However, PPO mitigates this by performing multiple optimization epochs on a single batch of collected data, making better use of the samples.
*   **Hyperparameter Sensitivity**: PPO, like most deep RL algorithms, is sensitive to hyperparameters. Key ones include:
    *   `clip_epsilon`: Controls the clipping range. Too small, and learning might be too slow; too large, and stability might suffer.
    *   `ppo_epochs`: Number of times to iterate over the collected data for optimization. More epochs mean better data utilization but can also lead to overfitting if too high.
    *   `learning rates` (for actor and critic): Crucial for convergence speed and stability.
    *   `gamma`: Discount factor, balancing immediate vs. future rewards.
*   **Computational Cost**: PPO involves collecting a batch of experiences, then running multiple optimization epochs on that batch. This can be computationally intensive, especially with large models and environments. However, its robustness often justifies the cost.

### Typical Use Cases and Modern Relevance

Actor-Critic methods, especially PPO, are incredibly versatile and are applied across a wide range of domains in 2026:

*   **Robotics**: For continuous control tasks, PPO is a go-to algorithm for teaching robots complex manipulation skills, locomotion, and navigation.
*   **Game AI**: Developing intelligent agents for complex video games, from strategy games to first-person shooters, where PPO agents can learn human-like or superhuman strategies.
*   **Autonomous Systems**: In self-driving cars or drone control, PPO can optimize control policies for various scenarios, though safety-critical applications often combine it with formal verification.
*   **Large Language Model (LLM) Alignment (RLHF)**: This is perhaps PPO's most impactful application in the current AI landscape. PPO is the core algorithm used in Reinforcement Learning from Human Feedback (RLHF) pipelines to align LLMs with human values and instructions. After an LLM is pre-trained and fine-tuned, a reward model (often a fine-tuned LLM itself) provides scalar feedback on generated text. PPO then optimizes the LLM's policy (its text generation probabilities) to maximize this reward, ensuring the model produces helpful, harmless, and honest outputs. This is the mechanism behind models like OpenAI's ChatGPT and Meta's Llama 2.

Understanding PPO is therefore not just about solving simple control problems, but about grasping a fundamental technique that underpins the alignment and advanced capabilities of the most sophisticated AI models today.


### Resources for Further Learning

*   **PyTorch Documentation**: The official PyTorch documentation for `torch.nn`, `torch.optim`, and `torch.distributions` is invaluable for understanding the building blocks of the agent.
    *   [PyTorch Docs](https://pytorch.org/docs/stable/index.html)
*   **Gymnasium Documentation**: Learn more about creating and interacting with various reinforcement learning environments.
    *   [Gymnasium Docs](https://gymnasium.farama.org/)
*   **Proximal Policy Optimization (PPO) Paper**: The original research paper by John Schulman et al. (2017) that introduced PPO.
    *   [PPO Paper on arXiv](https://arxiv.org/abs/1707.06347)
*   **Advantage Actor-Critic (A2C/A3C) Paper**: The original paper by Volodymyr Mnih et al. (2016) on Asynchronous Advantage Actor-Critic.
    *   [A3C Paper on arXiv](https://arxiv.org/abs/1602.01783)
*   **Hugging Face `trl` Library**: For practical applications of PPO in RLHF, especially with LLMs, the `trl` (Transformer Reinforcement Learning) library is a must-know.
    *   [Hugging Face `trl` GitHub](https://github.com/huggingface/trl)
    *   [Hugging Face `trl` Documentation](https://huggingface.co/docs/trl/index)
*   **Google AI Blog / DeepMind Publications**: Stay updated with the latest research and applications of RL, including Actor-Critic methods, from leading AI labs.
    *   [Google AI Blog](https://ai.googleblog.com/)
    *   [DeepMind Publications](https://deepmind.google/discover/blog/)
